In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/同花顺')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
print(f'股票数量:{len(codes)},第一支股票:{codes[0]}')
code = codes[0]
dt = datasource.getData(code)
dt.head()

股票数量:6974,第一支股票:000001


,Id,Code,Date,Open,High,Low,Close,Amount,Volume
0,1,000001,2014-01-02 00:00:00,12.12,12.30,12.05,12.23,596223740.0,48991089.0
1,2,000001,2014-01-03 00:00:00,12.15,12.16,11.78,11.93,656631300.0,55111484.0
2,3,000001,2014-01-06 00:00:00,11.89,12.00,11.50,11.67,679280380.0,58211823.0
3,4,000001,2014-01-07 00:00:00,11.53,11.76,11.51,11.63,393977580.0,33840749.0
4,5,000001,2014-01-08 00:00:00,11.64,11.95,11.53,11.76,538436170.0,45776816.0


# 前言
我想看看多日涨幅超过某个阈值的都是什么样子。

In [2]:
# 单只股票

In [3]:
DAYS = 20 # 多少个交易日
THRESHOLD = 100 # 超过某个阈值的
dt[f'next{DAYS}close'] = dt['Close'].shift(-DAYS) # 多少个交易日后的收盘价
dt[f'next{DAYS}closeRate'] = (dt[f'next{DAYS}close']/dt['Close']-1)*100 # 涨幅百分比
dt2 = dt.loc[dt[f'next{DAYS}closeRate']> THRESHOLD, :] # 超过某个阈值的


In [4]:
len(dt2)

0

# 全部股票

In [5]:
dts = []
for i in tqdm(range(len(codes))):
    code = codes[i]
    dt = datasource.getData(code, '2024-01-01')
    dt[f'next{DAYS}close'] = dt['Close'].shift(-DAYS) # 多少个交易日后的收盘价
    dt[f'next{DAYS}closeRate'] = (dt[f'next{DAYS}close']/dt['Close']-1)*100 # 涨幅百分比
    dt2 = dt.loc[dt[f'next{DAYS}closeRate']> THRESHOLD, :] # 超过某个阈值的
    # 看看有没有数据，
    if len(dt2) > 0:
        dts.append(dt2)

100%|█████████████████████████████████████████████████████████████████████████████| 6974/6974 [00:34<00:00, 200.75it/s]


In [6]:
len(dts)

518

In [7]:
dt_all = pd.concat(dts)
len(dt_all)

3483

In [8]:
dt_all.to_excel(f'{DAYS}天超过{THRESHOLD}.xlsx')